In [19]:
import os
import re
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import Ollama

In [20]:
class Config:
    EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
    LLM_MODEL = "llama3.2:3b"
    TEMPERATURE = 0.3  
    TEMPERATUREC = 0.5
    CHUNK_SIZE = 100  
    CHUNK_OVERLAP = 50  
    PERSIST_DIR = "./chroma_db_pt"
    PDF_PATHS = [
        r"M:\\tcc\\pdfs\\Guia Cores RGB.pdf"]

In [ ]:
# Função de limpeza 
def clean_Memoriaspost(text: str) -> str:
    patterns = [
        r"Diário Oficial .+? Página \d+",
        r"Lei Nº \d+\.\d+ de \d{2}/\d{2}/\d{4}",
        r"Publicado em: \d{2}/\d{2}/\d{4}",
        r"Este texto não substitui o original publicado",
        r"\n\s*\d+\s*\n"
    ]
    
    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    
    text = re.sub(r"(?i)(artigo|art\.) ?(\d+)", r"Art. \2", text)
    text = re.sub(r"§ ?(único|\d+º?)", r"§ \1", text)
    
    return re.sub(r"\s+", " ", text).strip()

# Carregar e processar documentos
documents = []
for path in Config.PDF_PATHS:
    try:
        loader = PyPDFLoader(path)
        pages = loader.load_and_split(text_splitter=None)
        for page in pages:
            cleaned = clean_Memoriaspost(page.page_content)
            if cleaned.strip():
                page.page_content = cleaned
                documents.append(page)
        print(f"{os.path.basename(path)} processado")
    except Exception as e:
        print(f"Erro em {path}: {str(e)}")

Ignoring wrong pointing object 753 0 (offset 0)


✓ Guia Cores RGB.pdf processado


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=Config.CHUNK_SIZE,
    chunk_overlap=Config.CHUNK_OVERLAP,
    separators=[
        r"\n\nArt\. \d+\.",  
        r"\n§ ",             
        r"\n\n", 
        r"\n", 
        " ", 
        ""
    ],
    length_function=lambda x: len(x.split()),
    is_separator_regex=False  
)

# Processamento de documentos
texts = text_splitter.split_documents(documents)

# Verificar duplicatas
seen = set()
duplicates = 0
for t in texts:
    h = hash(t.page_content.strip().lower())
    if h in seen:
        duplicates += 1
    seen.add(h)
print(f"Chunks duplicados: {duplicates}")

# Inicializar embeddings
embeddings = HuggingFaceEmbeddings(model_name=Config.EMBEDDING_MODEL)

db = Chroma.from_documents(
    texts,
    embeddings,
    persist_directory=Config.PERSIST_DIR,
    collection_metadata={"hnsw:space": "cosine"}
)

In [ ]:
# Configurar retriever
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 9,
        "lambda_mult": 0.45,
        "score_threshold": 0.5
    }
)

# Inicializar LLM
llm_cor = Ollama(
    model=Config.LLM_MODEL,
    temperature=Config.TEMPERATURE,
    system="Você é um robo especialista em cores RGB e que so responde no formato JSON fornecido melo usuario"
)

# Prompt template
prompt_template_cor = """
Você é um sistema responsável por interpretar comandos para controle de LEDs RGB. Seu trabalho é identificar:

1. Qual LED deve ser aceso (valores válidos: 1, 2 ou 3)
2. Qual cor RGB será usada (formato: (x, y, z), onde x, y e z são inteiros entre 0 e 255)

Use as informações a seguir, o codigo das cores esta logo depois do nome delas:

Contexto:
{context}

Pergunta:
{question}

Regras obrigatórias:
- A cor deve ser representada **somente no formato RGB** como uma tupla (x, y, z). **Nunca use hexadecimal**, nomes de cores ou outros formatos.
- Se a pergunta contiver **cor e número do LED**, retorne ambos corretamente.
- Se contiver **apenas a cor**, retorne "led": "null" e o RGB correspondente.
- Se contiver **apenas o LED**, retorne "rgb": "null" e o número do LED.


Instruções importantes:
- **Não adivinhe** valores. Use "null" quando a informação não estiver presente ou for ambígua.
- Não forneça nenhuma explicação ou texto adicional.
- A resposta deve ser **exatamente no formato JSON abaixo**.

Formato de saída (exato):
{{
  "rgb": (x, y, z) ou null,
  "led": número do LED (1, 2 ou 3) ou null
}}
""" 


PROMPT_COR = PromptTemplate(
    template=prompt_template_cor,
    input_variables=["context", "question"]
)

# Criar cadeia QA
qa_chain_cor = RetrievalQA.from_chain_type(
    llm=llm_cor,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT_COR}
)

import ast
import json
import re

def consultar_cor(pergunta, tentativas=3):
    def hex_to_rgb(hex_str):
        hex_str = hex_str.lstrip("#")
        if len(hex_str) == 6 and all(c in "0123456789abcdefABCDEF" for c in hex_str):
            r = int(hex_str[0:2], 16)
            g = int(hex_str[2:4], 16)
            b = int(hex_str[4:6], 16)
            return (r, g, b)
        return None

    for tentativa in range(tentativas):
        try:
            resposta = qa_chain_cor({"query": pergunta})
            resposta_json = resposta["result"]
            print(resposta_json)

            # Corrige parênteses para colchetes, se necessário
            resposta_json = resposta_json.replace("(", "[").replace(")", "]")

            # Converte JSON para dicionário
            dados = json.loads(resposta_json)

            # Interpreta RGB
            rgb_raw = dados.get("rgb")
            rgb = None

            if isinstance(rgb_raw, str):
                if re.fullmatch(r"#?[0-9a-fA-F]{6}", rgb_raw.strip()):
                    rgb = hex_to_rgb(rgb_raw.strip())
                else:
                    try:
                        rgb_tuple = ast.literal_eval(rgb_raw)
                        if (
                            isinstance(rgb_tuple, (tuple, list)) and len(rgb_tuple) == 3 and
                            all(isinstance(v, int) and 0 <= v <= 255 for v in rgb_tuple)
                        ):
                            rgb = tuple(rgb_tuple)
                    except:
                        rgb = None
            elif isinstance(rgb_raw, (tuple, list)) and len(rgb_raw) == 3:
                if all(isinstance(v, int) and 0 <= v <= 255 for v in rgb_raw):
                    rgb = tuple(rgb_raw)

            # Interpreta LED
            led_raw = dados.get("led")
            if isinstance(led_raw, str) and led_raw.isdigit():
                led_raw = int(led_raw)

            if isinstance(led_raw, int) and led_raw in [1, 2, 3]:
                led = led_raw
            else:
                led = None

            return rgb, led

        except Exception as e:
            print(f"Tentativa {tentativa+1} falhou: {e}")

    raise ValueError("Erro ao interpretar a resposta da IA após 3 tentativas.")


In [25]:
# Prompt template
prompt_template_init = """
Você é um assistente que interpreta a intenção do usuário.

O usuário pode:
- Querer controlar o LED
- Querer controlar o motor
- Querer controlar o motor e o LED
- Ou apenas conversar, sem pedir nenhuma dessas ações

Analise a seguinte pergunta do usuário:

Pergunta:
{question}

Responda **somente com um JSON**, indicando o que o usuário deseja:
{{ "acao": "led" }} ou {{ "acao": "motor" }} ou {{ "acao": "nenhum" }} ou {{ "acao": "led_motor" }}
"""

PROMPT_INIT= PromptTemplate(
    template=prompt_template_init,
    input_variables=["question"]
)

# Inicializar LLM
llm_init = Ollama(
    model=Config.LLM_MODEL,
    temperature=Config.TEMPERATURE,
    system="Você irá analisar a necessidade do usuário e executar o que ele quiser."
)

def fazer_pergunta(pergunta):
    prompt_texto = PROMPT_INIT.format(question=pergunta)
    resposta = llm_init(prompt=prompt_texto)
    return resposta


In [ ]:
resposta = fazer_pergunta("oi")
print(resposta)

{ "acao": "nenhum" }


In [ ]:
prompt_template_motor = """
Você é um assistente que ajusta a velocidade de um motor conforme o pedido do usuário.

Pedido do usuário:
{question}

Velocidade atual do motor:
{velocidade}

REGRAS:
1. Se o usuário dizer para "aumentar" (sem valor específico), adicione 0.5 à velocidade atual.
2. Se dizr para "diminuir" (sem valor específico), subtraia 0.5 da velocidade atual.
3. Se o usuário especificar um número, use exatamente o valor indicado.
4. Se o usuário dizer para "parar", defina a velocidade como 0.
5. Se dizer para "ligar" e não especificar velocidade, defina como 1.
6. A velocidade deve ficar entre 0 e 50. Se sair desse intervalo, ajuste para o mais próximo (mínimo 0, máximo 50).

ATENÇÃO: Sua resposta deve ser SOMENTE um JSON com **as aspas exatamente como nos exemplos abaixo**, sem NENHUM outro texto ou explicação.

EXEMPLOS VÁLIDOS:
{{
  "velocidade": "2"
}}

{{
  "velocidade": "2.5"
}}

Responda agora com o JSON correto.
"""


PROMPT_MOTOR = PromptTemplate(
    template=prompt_template_motor,
    input_variables=["question", "velocidade"]
)
llm_motor = Ollama(
    model=Config.LLM_MODEL,
    temperature=Config.TEMPERATURE,
    system="Você ira analisar a frase do usuario e celecionar a velocidade adequada para satisfaze-lo."
)

def consultar_motor(pergunta, velocidade_atual, tentativas=0):
    if tentativas >= 3:
        raise ValueError("Não foi possível obter uma velocidade válida após 3 tentativas.")
    
    prompt_texto = PROMPT_MOTOR.format(question=pergunta, velocidade=velocidade_atual)
    resposta = llm_motor(prompt_texto)
    
    # Corrigir a resposta para garantir aspas no número
    resposta_corrigida = re.sub(r'("velocidade"\s*:\s*)(\d+\.?\d*)', r'\1"\2"', resposta)

    try:
        resultado = json.loads(resposta_corrigida)
        valor_str = resultado.get("velocidade")
        valor = float(valor_str)

        # Limita entre 0 e 50
        valor = max(0, min(50, valor))
        return valor
    
    except (json.JSONDecodeError, ValueError, TypeError):
        return consultar_motor(pergunta, velocidade_atual, tentativas + 1)


In [28]:
consultar_motor("aumente a velocidade",1)

1.5

In [29]:
def executar_acao(pergunta, resposta_json, velocidade_motor_atual, painel):
    resultado = {
        "velocidade": None,
        "led": None,
        "rgb": None
    }

    try:
        dados = json.loads(resposta_json)
        acao = dados.get("acao")
        
        # === SEM AÇÃO ===
        if acao == "nenhum":
            painel.append_to_ia_output(
                "Você pode:\n"
                "- Acender o LED 1, 2 ou 3 com qualquer cor no formato RGB (ex: 'acenda o led 1 de vermelho').\n"
                "- Ligar ou desligar o motor.\n"
                "- Aumentar ou diminuir a velocidade do motor.\n"
                "- Definir uma velocidade específica para o motor (ex: 'motor para 10 rps')."
            )
            return resultado
        
        # === MOTOR ===
        if acao in ["motor", "led_motor"]:
            resultado["velocidade"]  = consultar_motor(pergunta, velocidade_motor_atual)
            
            

        # === LED ===
        if acao in ["led", "led_motor"]:
            resultado["rgb"] , resultado["led"] = consultar_cor(pergunta)

        return resultado

    except json.JSONDecodeError:
        painel.append_to_ia_output("Erro: resposta da IA mal formatada.")
        raise


In [ ]:
import tkinter as tk
from tkinter import ttk
import threading
import time
import math

class PainelControle:
    def __init__(self, root):
        self.root = root
        self.root.title("Painel de Controle com IA")
        self.root.configure(bg="#1e1e2f")

        style = ttk.Style()
        style.theme_use("clam")
        style.configure("TEntry", foreground="white", fieldbackground="#2e2e3e", background="#2e2e3e")

        # Histórico de comandos
        self.command_history = []
        self.history_index = -1

        # Estrutura principal
        self.main_frame = tk.Frame(root, bg="#1e1e2f")
        self.main_frame.pack(padx=10, pady=10)

        # Painel de controle (esquerda)
        self.control_frame = tk.Frame(self.main_frame, bg="#1e1e2f")
        self.control_frame.grid(row=0, column=0, padx=10)

        # LEDs
        self.leds = {}
        led_frame = tk.Frame(self.control_frame, bg="#1e1e2f")
        led_frame.grid(row=0, column=0, columnspan=3, pady=10)
        for i in range(1, 4):
            led = tk.Canvas(led_frame, width=30, height=30, bg='gray', highlightthickness=0)
            led.grid(row=0, column=i-1, padx=15)
            self.leds[f'led{i}'] = led

        # Motor
        self.motor_speed = 0
        self.angle = 0
        self.motor_canvas = tk.Canvas(self.control_frame, width=120, height=120, bg="#1e1e2f", highlightthickness=0)
        self.motor_canvas.grid(row=1, column=0, columnspan=3, pady=20)
        self.motor_circle = self.motor_canvas.create_oval(20, 20, 100, 100, fill="black", outline="white")
        self.pointer = self.motor_canvas.create_line(60, 60, 60, 30, fill="red", width=4)

        # Caixa de entrada
        self.command_entry = tk.Text(self.control_frame, width=40, height=4, wrap="word", bg="#2e2e3e", fg="white", insertbackground="white")
        self.command_entry.grid(row=2, column=0, columnspan=3, pady=10)
        self.command_entry.bind("<Return>", self.on_enter)
        self.command_entry.bind("<Up>", self.on_up_arrow)
        self.command_entry.bind("<Down>", self.on_down_arrow)

        # Caixa de saída 
        self.ia_output = tk.Text(self.main_frame, width=40, height=20, bg="#12121c", fg="white", wrap="word", state="disabled")
        self.ia_output.grid(row=0, column=1, sticky="n")

        # Animação do motor
        self.running = True
        threading.Thread(target=self.animate_motor, daemon=True).start()

    def set_led(self, led_id, rgb):
        if led_id in self.leds:
            color = f'#{rgb[0]:02x}{rgb[1]:02x}{rgb[2]:02x}'
            self.leds[led_id].configure(bg=color)

    def set_motor_speed(self, speed):
        self.motor_speed = speed

    def animate_motor(self):
        frame_time = 0.05  
        while self.running:
            if self.motor_speed > 0:
                self.angle = (self.angle + 360 * self.motor_speed * frame_time) % 360
                rad = math.radians(self.angle)
                x = 60 + 30 * math.sin(rad)
                y = 60 - 30 * math.cos(rad)
                self.motor_canvas.coords(self.pointer, 60, 60, x, y)
            time.sleep(frame_time)


    def on_enter(self, event):
        texto = self.command_entry.get("1.0", tk.END).strip()
        self.command_entry.delete("1.0", tk.END)

        if texto:
            self.command_history.append(texto)
            self.history_index = len(self.command_history)

        self.callIA(texto)
        return "break"

    def on_up_arrow(self, event):
        if self.command_history and self.history_index > 0:
            self.history_index -= 1
            comando = self.command_history[self.history_index]
            self.command_entry.delete("1.0", tk.END)
            self.command_entry.insert(tk.END, comando)
        return "break"

    def on_down_arrow(self, event):
        if self.command_history and self.history_index < len(self.command_history) - 1:
            self.history_index += 1
            comando = self.command_history[self.history_index]
            self.command_entry.delete("1.0", tk.END)
            self.command_entry.insert(tk.END, comando)
        else:
            self.command_entry.delete("1.0", tk.END)
            self.history_index = len(self.command_history)
        return "break"

    def append_to_ia_output(self, message):
        self.ia_output.configure(state="normal")
        self.ia_output.insert(tk.END, message + "\n")
        self.ia_output.see(tk.END)
        self.ia_output.configure(state="disabled")

    def callIA(self, texto):
        texto = texto.lower().strip()
        
        resposta = fazer_pergunta(texto)
        resultado = executar_acao(texto, resposta, self.motor_speed, self)

        led = resultado.get("led")
        rgb = resultado.get("rgb")

        if led is not None and rgb is not None:
            self.set_led(f"led{led}", rgb)
            self.append_to_ia_output(f"led{led} ligado com cor RGB {rgb}.")

        elif led is not None and rgb is None:
            self.append_to_ia_output(
                f"Entendemos que você quer ligar o LED {led}. "
                f"Por favor, informe também a cor em formato RGB (ex: 'vermelho', 'RGB 255 0 0')."
            )

        elif rgb is not None and led is None:
            self.append_to_ia_output(
                f"Entendemos que você quer ligar um LED com a cor {rgb}. "
                f"Por favor, informe também qual número do LED deseja acender (1, 2 ou 3)."
            )

        velocidade = resultado.get("velocidade")
        if velocidade is not None:
            try:
                velocidade_float = float(velocidade)
                self.set_motor_speed(velocidade_float)
                self.append_to_ia_output(f"Motor ajustado para {velocidade_float} rotações por segundo.")
            except ValueError:
                self.append_to_ia_output("Erro ao interpretar a velocidade do motor.")


        


if __name__ == '__main__':
    root = tk.Tk()
    painel = PainelControle(root)
    root.mainloop()


{
  "rgb": (255, 0, 0),
  "led": 1
}
